# CVE Spectral Drift — Period Comparison

This notebook demonstrates **spectral drift** between two CVE embedding corpora:
- **Period A**: CVE-1999 → CVE-2014 (`embs_99_to_14.npy`)
- **Period B**: CVE-2015 → CVE-2025 (`embs_15_to_2025.npy`)

The analysis builds a k-NN graph for each period, computes the **graph Laplacian spectrum**,
and quantifies the drift between the two spectral distributions using:
- Wasserstein-1 distance on eigenvalue distributions
- Frobenius norm between normalised Laplacians (optional)
- Visual overlay of spectral densities (KDE)

> **Data scientist note**: the embeddings were produced by a sentence-transformer model
> fine-tuned on NVD descriptions. Each row is one CVE entry; the embedding dimension is 384.

> **Sections 1-7** work standalone (no server needed).
> **Section 8** calls the live `/api/drift/*` endpoints — start the server first with
> `uv run src/arro_server`.

In [ ]:
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spla
from scipy.stats import wasserstein_distance, gaussian_kde
from sklearn.neighbors import kneighbors_graph
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path

ROOT = Path.cwd().parent
DATA = ROOT / 'data' / 'cve_embeddings_demo'
RESULTS = ROOT / 'data' / 'results'
RESULTS.mkdir(parents=True, exist_ok=True)

K = 10        # neighbours for kNN graph
N_EIGS = 200  # Laplacian eigenvalues to compute (smallest non-zero)
N_SAMPLE = 5_000  # subsample per period (increase if RAM > 32 GB)
SEED = 42

## 1. Load Embeddings

In [ ]:
embs_A_raw = np.load(DATA / 'embs_99_to_14.npy', allow_pickle=True)
embs_B_raw = np.load(DATA / 'embs_15_to_2025.npy', allow_pickle=True)

def _coerce(arr):
    if arr.dtype == object:
        return np.vstack(arr.tolist()).astype(np.float32)
    return arr.astype(np.float32)

embs_A = _coerce(embs_A_raw)
embs_B = _coerce(embs_B_raw)

print(f'Period A (1999-2014): {embs_A.shape}  ({embs_A.nbytes / 1e6:.0f} MB)')
print(f'Period B (2015-2025): {embs_B.shape}  ({embs_B.nbytes / 1e6:.0f} MB)')

## 2. Subsample for Tractable Laplacian Computation

Computing a Laplacian on the full corpus (100k+ entries) is expensive.
We draw a **stratified random sample** of `N_SAMPLE` entries per period.
Increase `N_SAMPLE` if you have > 32 GB RAM.

In [ ]:
rng = np.random.default_rng(SEED)
idx_A = rng.choice(len(embs_A), size=min(N_SAMPLE, len(embs_A)), replace=False)
idx_B = rng.choice(len(embs_B), size=min(N_SAMPLE, len(embs_B)), replace=False)
X_A = embs_A[idx_A]
X_B = embs_B[idx_B]
print(f'Sample A: {X_A.shape}, Sample B: {X_B.shape}')

## 3. Build Symmetric k-NN Graph & Normalised Graph Laplacian

We use the **symmetric normalised Laplacian**:

$$L_{\text{sym}} = I - D^{-1/2} A D^{-1/2}$$

whose eigenvalues lie in [0, 2]. A spectral shift between periods
indicates that the geometry of the semantic neighbourhood structure has changed.

In [ ]:
def build_normalised_laplacian(X, k=K):
    A = kneighbors_graph(X, n_neighbors=k, mode='connectivity',
                         include_self=False, n_jobs=-1)
    A = (A + A.T).astype(np.float32)
    A.data[:] = 1.0  # binarise
    d = np.asarray(A.sum(axis=1)).flatten()
    d_inv_sqrt = np.where(d > 0, 1.0 / np.sqrt(d), 0.0)
    D_inv_sqrt = sp.diags(d_inv_sqrt)
    L = sp.eye(X.shape[0]) - D_inv_sqrt @ A @ D_inv_sqrt
    return L.tocsr()

print('Building Laplacian A ...')
L_A = build_normalised_laplacian(X_A)
print('Building Laplacian B ...')
L_B = build_normalised_laplacian(X_B)
print('Done.')

## 4. Compute Eigenvalue Spectra

In [ ]:
def compute_spectrum(L, k=N_EIGS):
    k = min(k, L.shape[0] - 2)
    eigs, _ = spla.eigsh(L, k=k, which='SM', tol=1e-4, maxiter=3000)
    return np.sort(np.real(eigs))

print('Computing spectrum A ...')
eigs_A = compute_spectrum(L_A)
print('Computing spectrum B ...')
eigs_B = compute_spectrum(L_B)
print(f'Spectrum A: min={eigs_A.min():.4f}, max={eigs_A.max():.4f}')
print(f'Spectrum B: min={eigs_B.min():.4f}, max={eigs_B.max():.4f}')

## 5. Drift Metrics

In [ ]:
w1 = wasserstein_distance(eigs_A, eigs_B)
gap_A = float(eigs_A[eigs_A > 1e-6][0]) if np.any(eigs_A > 1e-6) else 0.0
gap_B = float(eigs_B[eigs_B > 1e-6][0]) if np.any(eigs_B > 1e-6) else 0.0
mean_shift = float(np.mean(eigs_B) - np.mean(eigs_A))
interpretation = 'low' if w1 < 0.01 else 'medium' if w1 < 0.05 else 'high'

print('=== Spectral Drift Metrics ===')
print(f'Wasserstein-1 (W₁):          {w1:.6f}  [{interpretation}]')
print(f'Spectral gap A (1999-2014):  {gap_A:.6f}')
print(f'Spectral gap B (2015-2025):  {gap_B:.6f}')
print(f'Mean eigenvalue shift (B-A): {mean_shift:+.6f}')

## 6. Visualise Spectral Density (KDE Overlay)

In [ ]:
fig = plt.figure(figsize=(16, 10))
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.35)

# --- KDE overlay ---
x_grid = np.linspace(0, 2, 500)
kde_A = gaussian_kde(eigs_A, bw_method='scott')
kde_B = gaussian_kde(eigs_B, bw_method='scott')

ax1 = fig.add_subplot(gs[0, 0])
ax1.fill_between(x_grid, kde_A(x_grid), alpha=0.35, color='steelblue', label='Period A (1999-2014)')
ax1.fill_between(x_grid, kde_B(x_grid), alpha=0.35, color='tomato', label='Period B (2015-2025)')
ax1.plot(x_grid, kde_A(x_grid), color='steelblue', lw=1.5)
ax1.plot(x_grid, kde_B(x_grid), color='tomato', lw=1.5)
ax1.set_title('Spectral Density — KDE Overlay')
ax1.set_xlabel('Eigenvalue λ')
ax1.set_ylabel('Density')
ax1.legend(fontsize=9)
ax1.annotate(f'W₁ = {w1:.4f}  [{interpretation}]', xy=(0.52, 0.88),
             xycoords='axes fraction', fontsize=10,
             bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.8))

# --- Sorted eigenvalue rank plot ---
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(eigs_A, color='steelblue', lw=1.5, label='Period A')
ax2.plot(eigs_B, color='tomato', lw=1.5, label='Period B')
ax2.set_title('Sorted Eigenvalues (rank plot)')
ax2.set_xlabel('Rank')
ax2.set_ylabel('Eigenvalue λ')
ax2.legend(fontsize=9)

# --- Spectral difference curve ---
ax3 = fig.add_subplot(gs[1, 0])
n_common = min(len(eigs_A), len(eigs_B))
diff = eigs_B[:n_common] - eigs_A[:n_common]
ax3.fill_between(range(n_common), diff, alpha=0.5,
                  color='mediumpurple' if diff.mean() > 0 else 'darkorange')
ax3.axhline(0, color='gray', lw=0.8, ls='--')
ax3.set_title('Eigenvalue Difference (B − A, by rank)')
ax3.set_xlabel('Rank')
ax3.set_ylabel('λ_B − λ_A')

# --- Summary text ---
ax4 = fig.add_subplot(gs[1, 1])
ax4.axis('off')
summary = (
    f'Drift Summary\n'
    f'──────────────────────────\n'
    f'W₁ distance:     {w1:.6f}\n'
    f'Interpretation:  {interpretation.upper()}\n'
    f'Spectral gap A:  {gap_A:.6f}\n'
    f'Spectral gap B:  {gap_B:.6f}\n'
    f'Mean shift (B-A): {mean_shift:+.6f}\n'
    f'\nSample size: {N_SAMPLE:,} per period\n'
    f'k-neighbours: {K}  |  n_eigs: {N_EIGS}'
)
ax4.text(0.1, 0.9, summary, transform=ax4.transAxes,
         fontsize=10, verticalalignment='top',
         fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='#f5f5f5', alpha=0.8))

plt.suptitle('CVE Corpus — Graph Laplacian Spectral Drift', fontsize=14, fontweight='bold', y=1.01)
plt.savefig(RESULTS / 'spectral_drift_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Plot saved to {RESULTS / "spectral_drift_overview.png"}')

## 7. Interpretation Guide

| Metric | What it measures | Expected direction |
|--------|------------------|--------------------|
| Wasserstein-1 (W₁) | Earth-mover distance between spectral distributions | Higher = more drift |
| Spectral gap λ₂ | Connectivity / cluster separation of the kNN graph | Lower in B → denser, less modular community structure |
| Mean eigenvalue shift | Average shift of the whole spectrum | Positive → B is more connected on average |

A large W₁ together with a shrinking spectral gap in Period B
suggests that modern CVEs form **denser semantic clusters** —
i.e., vulnerability language has become more specialised and topic-concentrated
(e.g. cloud-native, supply-chain, ML/AI categories emerging post-2015).

### Interpretation thresholds

| W₁ range | Label | Meaning |
|----------|-------|---------|
| < 0.01 | **low** | Spectra are almost identical — minimal drift |
| 0.01 – 0.05 | **medium** | Measurable drift — worth monitoring |
| > 0.05 | **high** | Significant drift — consider retraining the downstream model |

## 8. Live API Demo

The cells below call the running `arro-server` to show the same metrics
served over HTTP — no Python data loading required on the client side.

Start the server first:
```bash
uv run src/arro_server
```

In [ ]:
import requests

BASE = 'http://localhost:8000'

def get(path, **kwargs):
    r = requests.get(f'{BASE}{path}', **kwargs)
    r.raise_for_status()
    return r.json()

def post(path, **kwargs):
    r = requests.post(f'{BASE}{path}', **kwargs)
    r.raise_for_status()
    return r.json()

In [ ]:
# Check drift engine readiness
health = get('/api/drift/health')
print('Drift engine health:', health)

In [ ]:
# Fetch the live scalar drift score
score = get('/api/drift/score')
print(f"Live W₁ drift score: {score['drift_score']:.6f}  [{score['interpretation']}]")
print(f"  Period A: {score['period_a']}  ({score['period_a_n_lambdas']} eigenvalues)")
print(f"  Period B: {score['period_b']}  ({score['period_b_n_lambdas']} eigenvalues)")

In [ ]:
# Fetch full eigenvalue arrays from the server and plot
lams = get('/api/drift/lambdas')
eigs_api_A = np.array(lams['period_a']['lambdas'])
eigs_api_B = np.array(lams['period_b']['lambdas'])

fig, ax = plt.subplots(figsize=(10, 4))
x_grid = np.linspace(0, 2, 400)
if len(eigs_api_A) > 3:
    kde_a = gaussian_kde(eigs_api_A)
    ax.fill_between(x_grid, kde_a(x_grid), alpha=0.35, color='steelblue', label=f"API Period A (n={len(eigs_api_A)})")
if len(eigs_api_B) > 3:
    kde_b = gaussian_kde(eigs_api_B)
    ax.fill_between(x_grid, kde_b(x_grid), alpha=0.35, color='tomato', label=f"API Period B (n={len(eigs_api_B)})")
ax.set_title(f"Live API — Spectral Density (W₁={lams['drift_score']:.4f})")
ax.set_xlabel('Eigenvalue λ')
ax.set_ylabel('Density')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Side-by-side search: use a random query vector (replace with a real CVE embedding)
rng_api = np.random.default_rng(0)
query_vec = rng_api.standard_normal(embs_A.shape[1]).tolist()  # 384-d

result = post('/api/drift/search', json={'vector': query_vec, 'k': 5, 'tau': 0.5})

print(f"Drift score: {result['drift_score']:.6f}\n")
print('=== Period A results ===')
for r in result['period_a']['results']:
    print(' ', r)
print()
print('=== Period B results ===')
for r in result['period_b']['results']:
    print(' ', r)